In [1]:
from pathlib import Path

import json
import pandas as pd


PROJECT_ROOT = Path.cwd().parent

PROCESSED_ROOT = PROJECT_ROOT / "data" / "alfa" / "processed"
METADATA_ROOT = PROJECT_ROOT / "data" / "alfa" / "metadata"

inventory_path = METADATA_ROOT / "processed_flight_inventory.csv"
topic_assessment_path = METADATA_ROOT / "processed_dataset_topic_assessment.csv"
file_profile_path = METADATA_ROOT / "processed_dataset_file_profile.csv"
failure_status_inventory_path = METADATA_ROOT / "processed_failure_status_inventory.csv"
failure_status_signal_summary_path = METADATA_ROOT / "processed_failure_status_signal_summary.csv"

inventory_frame = pd.read_csv(inventory_path)
topic_assessment_frame = pd.read_csv(topic_assessment_path)
file_profile_frame = pd.read_csv(file_profile_path)
failure_status_inventory_frame = pd.read_csv(failure_status_inventory_path)
failure_status_signal_summary_frame = pd.read_csv(failure_status_signal_summary_path)

print(f"Flights loaded : {len(inventory_frame)}")
print(f"Distinct topics loaded : {topic_assessment_frame['topic_name'].nunique()}")


Flights loaded : 47
Distinct topics loaded : 39


In [2]:
inventory_frame[[
    "flight_name",
    "fault_label_raw",
    "csv_file_count",
    "topic_count",
    "failure_status_topic_count",
    "missing_timestamp_topic_count"
]].head(10)

,flight_name,fault_label_raw,csv_file_count,topic_count,failure_status_topic_count,missing_timestamp_topic_count
0,carbonZ_2018-07-18-12-10-11_no_ground_truth,no_ground_truth,33,33,0,0
1,carbonZ_2018-07-18-15-53-31_1_engine_failure,engine_failure,34,34,1,0
2,carbonZ_2018-07-18-15-53-31_2_engine_failure,engine_failure,34,34,1,0
3,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,engine_failure_with_emr_traj,35,35,1,0
4,carbonZ_2018-07-18-16-37-39_1_no_failure,no_failure,30,30,0,0
5,carbonZ_2018-07-18-16-37-39_2_engine_failure_w...,engine_failure_with_emr_traj,35,35,1,0
6,carbonZ_2018-07-30-16-29-45_engine_failure_wit...,engine_failure_with_emr_traj,35,35,1,0
7,carbonZ_2018-07-30-16-39-00_1_engine_failure,engine_failure,34,34,1,0
8,carbonZ_2018-07-30-16-39-00_2_engine_failure,engine_failure,34,34,1,0
9,carbonZ_2018-07-30-16-39-00_3_no_failure,no_failure,33,33,0,0


In [3]:
fault_distribution = (inventory_frame["fault_label_raw"]
                      .value_counts()
                      .sort_index()
                      .rename_axis("fault_label_raw")
                      .reset_index(name = "flight_count")
)

fault_distribution

,fault_label_raw,flight_count
0,both_ailerons_failure,1
1,elevator_failure,2
2,engine_failure,7
3,engine_failure_with_emr_traj,16
4,left_aileron__right_aileron__failure,1
5,left_aileron_failure,2
6,no_failure,10
7,no_ground_truth,1
8,right_aileron_failure,2
9,right_aileron_failure_with_emr_traj,1


In [4]:
topic_assessment_frame[["topic_name",
                        "flight_count",
                        "min_row_count",
                        "max_row_count",
                        "malformed_file_count",
                        "malformed_row_total",
                        "is_present_in_all_flights",
                        "has_stable_column_count",
                        "is_structurally_clean"
]].sort_values(
    ["is_structurally_clean", "flight_count", "topic_name"],
    ascending = [False, False, True]
).head(50)

,topic_name,flight_count,min_row_count,max_row_count,malformed_file_count,malformed_row_total,is_present_in_all_flights,has_stable_column_count,is_structurally_clean
2,mavros-battery.csv,47,543,4721,0,0,True,True,True
3,mavros-global_position-compass_hdg.csv,47,110,957,0,0,True,True,True
4,mavros-global_position-global.csv,47,110,957,0,0,True,True,True
5,mavros-global_position-local.csv,47,110,957,0,0,True,True,True
6,mavros-global_position-raw-fix.csv,47,132,1167,0,0,True,True,True
7,mavros-global_position-raw-gps_vel.csv,47,132,1167,0,0,True,True,True
8,mavros-global_position-rel_alt.csv,47,110,957,0,0,True,True,True
9,mavros-imu-atm_pressure.csv,47,264,2334,0,0,True,True,True
10,mavros-imu-data.csv,47,67,574,0,0,True,True,True
11,mavros-imu-data_raw.csv,47,264,2334,0,0,True,True,True


In [5]:
structurally_eligible_topics_frame = topic_assessment_frame.loc[
    (topic_assessment_frame["is_present_in_all_flights"] == True) &
    (topic_assessment_frame["has_stable_column_count"] == True) &
    (topic_assessment_frame["is_structurally_clean"] == True)
].copy()

structurally_eligible_topics_frame[["topic_name",
                       "flight_count",
                       "min_row_count",
                       "max_row_count"
]].sort_values(["flight_count", "topic_name"], ascending = [False, True])

,topic_name,flight_count,min_row_count,max_row_count
2,mavros-battery.csv,47,543,4721
3,mavros-global_position-compass_hdg.csv,47,110,957
4,mavros-global_position-global.csv,47,110,957
5,mavros-global_position-local.csv,47,110,957
6,mavros-global_position-raw-fix.csv,47,132,1167
7,mavros-global_position-raw-gps_vel.csv,47,132,1167
8,mavros-global_position-rel_alt.csv,47,110,957
9,mavros-imu-atm_pressure.csv,47,264,2334
10,mavros-imu-data.csv,47,67,574
11,mavros-imu-data_raw.csv,47,264,2334


In [6]:
topic_assessment_frame.loc[
    topic_assessment_frame["is_structurally_clean"] == False,
    ["topic_name", "flight_count", "malformed_file_count", "malformed_row_total"]
].sort_values(["malformed_row_total", "topic_name"], ascending = [False, True])

,topic_name,flight_count,malformed_file_count,malformed_row_total
1,mavlink-from.csv,47,47,717988
0,diagnostics.csv,47,2,55


In [7]:
failure_status_inventory_frame[["flight_name",
                                "fault_label_raw",
                                "topic_name",
                                "row_count",
                                "timestamp_column",
                                "timestamp_is_monotonic",
                                "parse_status"
]].sort_values(["flight_name", "topic_name"]).head(50)

,flight_name,fault_label_raw,topic_name,row_count,timestamp_column,timestamp_is_monotonic,parse_status
0,carbonZ_2018-07-18-15-53-31_1_engine_failure,engine_failure,failure_status-engines.csv,81,%time,True,well_formed
1,carbonZ_2018-07-18-15-53-31_2_engine_failure,engine_failure,failure_status-engines.csv,77,%time,True,well_formed
2,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,engine_failure_with_emr_traj,failure_status-engines.csv,80,%time,True,well_formed
3,carbonZ_2018-07-18-16-37-39_2_engine_failure_w...,engine_failure_with_emr_traj,failure_status-engines.csv,83,%time,True,well_formed
4,carbonZ_2018-07-30-16-29-45_engine_failure_wit...,engine_failure_with_emr_traj,failure_status-engines.csv,96,%time,True,well_formed
5,carbonZ_2018-07-30-16-39-00_1_engine_failure,engine_failure,failure_status-engines.csv,75,%time,True,well_formed
6,carbonZ_2018-07-30-16-39-00_2_engine_failure,engine_failure,failure_status-engines.csv,74,%time,True,well_formed
7,carbonZ_2018-07-30-17-10-45_engine_failure_wit...,engine_failure_with_emr_traj,failure_status-engines.csv,80,%time,True,well_formed
8,carbonZ_2018-07-30-17-20-01_engine_failure_wit...,engine_failure_with_emr_traj,failure_status-engines.csv,96,%time,True,well_formed
9,carbonZ_2018-07-30-17-36-35_engine_failure_wit...,engine_failure_with_emr_traj,failure_status-engines.csv,118,%time,True,well_formed


In [8]:
failure_status_signal_summary_frame.sort_values(
    ["non_zero_flight_count", "topic_name", "column_name"],
    ascending = [False, True, True]
)

,topic_name,column_name,flight_count,non_zero_flight_count,min_first_non_zero_time,max_first_non_zero_time,min_value,max_value
2,failure_status-engines.csv,field.data,23,23,1.531944e+18,1.539876e+18,1.0,1.0
0,failure_status-aileron.csv,field.data,8,8,1.536692e+18,1.538769e+18,1.0,3.0
3,failure_status-rudder.csv,field.data,4,4,1.536693e+18,1.536702e+18,1.0,3.0
1,failure_status-elevator.csv,field.data,2,2,1.536692e+18,1.536693e+18,1.0,1.0


In [9]:
flights_with_failure_status = set(failure_status_inventory_frame["flight_name"].unique().tolist())

inventory_frame["has_failure_status_file"] = inventory_frame["flight_name"].isin(flights_with_failure_status)

inventory_frame.groupby(
    ["fault_label_raw", "has_failure_status_file"],
    dropna = False
).size().reset_index(name = "flight_count").sort_values(
    ["fault_label_raw", "has_failure_status_file"]
)

,fault_label_raw,has_failure_status_file,flight_count
0,both_ailerons_failure,True,1
1,elevator_failure,True,2
2,engine_failure,True,7
3,engine_failure_with_emr_traj,True,16
4,left_aileron__right_aileron__failure,True,1
5,left_aileron_failure,True,2
6,no_failure,False,10
7,no_ground_truth,False,1
8,right_aileron_failure,True,2
9,right_aileron_failure_with_emr_traj,True,1


In [10]:
fault_flights = inventory_frame.loc[
    ~inventory_frame["fault_label_raw"].isin(["no_failure", "no_ground_truth"])
].copy()

normal_flights = inventory_frame.loc[
    inventory_frame["fault_label_raw"] == "no_failure"
].copy()

unknown_ground_truth_flights = inventory_frame.loc[
    inventory_frame["fault_label_raw"] == "no_ground_truth"
].copy()

flights_with_failure_status = set(failure_status_inventory_frame["flight_name"].unique().tolist())

cohort_summary = pd.DataFrame(
    [
        {
            "cohort_name": "normal_flights",
            "flight_count": int(len(normal_flights))
        },
        {
            "cohort_name": "fault_flights",
            "flight_count": int(len(fault_flights))
        },
        {
            "cohort_name": "fault_flights_with_failure_status",
            "flight_count": int(fault_flights["flight_name"].isin(flights_with_failure_status).sum())
        },
        {
            "cohort_name": "unknown_ground_truth_flights",
            "flight_count": int(len(unknown_ground_truth_flights))
        }
    ]
)

cohort_summary

,cohort_name,flight_count
0,normal_flights,10
1,fault_flights,36
2,fault_flights_with_failure_status,36
3,unknown_ground_truth_flights,1


In [11]:
topic_fault_coverage_frame = (
    file_profile_frame
    .groupby(["topic_name", "fault_label_raw"], as_index = False)
    .agg(
        flight_count = ("flight_name", "nunique"),
        malformed_file_count = ("malformed_row_count", lambda s: int((s > 0).sum()))
    )
    .sort_values(["topic_name", "fault_label_raw"])
    .reset_index(drop = True)
)

topic_fault_coverage_frame.head(100)

,topic_name,fault_label_raw,flight_count,malformed_file_count
0,diagnostics.csv,both_ailerons_failure,1,0
1,diagnostics.csv,elevator_failure,2,0
2,diagnostics.csv,engine_failure,7,1
3,diagnostics.csv,engine_failure_with_emr_traj,16,0
4,diagnostics.csv,left_aileron__right_aileron__failure,1,0
...,...,...,...,...
95,mavros-global_position-global.csv,left_aileron_failure,2,0
96,mavros-global_position-global.csv,no_failure,10,0
97,mavros-global_position-global.csv,no_ground_truth,1,0
98,mavros-global_position-global.csv,right_aileron_failure,2,0


In [12]:
structurally_eligible_topics_frame[["topic_name",
                                    "flight_count",
                                    "min_row_count",
                                    "max_row_count",
                                    "duplicate_rows_total"
]].sort_values(["topic_name"]).reset_index(drop = True)

,topic_name,flight_count,min_row_count,max_row_count,duplicate_rows_total
0,mavros-battery.csv,47,543,4721,0.0
1,mavros-global_position-compass_hdg.csv,47,110,957,0.0
2,mavros-global_position-global.csv,47,110,957,0.0
3,mavros-global_position-local.csv,47,110,957,0.0
4,mavros-global_position-raw-fix.csv,47,132,1167,0.0
5,mavros-global_position-raw-gps_vel.csv,47,132,1167,0.0
6,mavros-global_position-rel_alt.csv,47,110,957,0.0
7,mavros-imu-atm_pressure.csv,47,264,2334,0.0
8,mavros-imu-data.csv,47,67,574,0.0
9,mavros-imu-data_raw.csv,47,264,2334,0.0


In [13]:
topic_screening_frame = topic_assessment_frame.copy()

topic_screening_frame["is_structurally_eligible"] = (
    topic_screening_frame["is_present_in_all_flights"] == True
) & (
    topic_screening_frame["has_stable_column_count"] == True
) & (
    topic_screening_frame["is_structurally_clean"] == True
)

topic_screening_frame[["topic_name",
                       "flight_count",
                       "min_column_count",
                       "max_column_count",
                       "malformed_file_count",
                       "malformed_row_total",
                       "is_present_in_all_flights",
                       "has_stable_column_count",
                       "is_structurally_clean",
                       "is_structurally_eligible"
]].sort_values(
    ["is_structurally_eligible", "topic_name"],
    ascending = [False, True]
).reset_index(drop = True)

,topic_name,flight_count,min_column_count,max_column_count,malformed_file_count,malformed_row_total,is_present_in_all_flights,has_stable_column_count,is_structurally_clean,is_structurally_eligible
0,mavros-battery.csv,47,16,16,0,0,True,True,True,True
1,mavros-global_position-compass_hdg.csv,47,2,2,0,0,True,True,True,True
2,mavros-global_position-global.csv,47,19,19,0,0,True,True,True,True
3,mavros-global_position-local.csv,47,90,90,0,0,True,True,True,True
4,mavros-global_position-raw-fix.csv,47,19,19,0,0,True,True,True,True
5,mavros-global_position-raw-gps_vel.csv,47,10,10,0,0,True,True,True,True
6,mavros-global_position-rel_alt.csv,47,2,2,0,0,True,True,True,True
7,mavros-imu-atm_pressure.csv,47,6,6,0,0,True,True,True,True
8,mavros-imu-data.csv,47,41,41,0,0,True,True,True,True
9,mavros-imu-data_raw.csv,47,41,41,0,0,True,True,True,True


In [14]:
fault_label_reference = pd.DataFrame(
    [
        {"fault_label_raw": "both_ailerons_failure", "fault_family": "aileron", "fault_label_grouped": "both_ailerons_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "elevator_failure", "fault_family": "elevator", "fault_label_grouped": "elevator_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "engine_failure", "fault_family": "engine", "fault_label_grouped": "engine_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "engine_failure_with_emr_traj", "fault_family": "engine", "fault_label_grouped": "engine_failure", "recovery_variant": "emr_traj"},
        {"fault_label_raw": "left_aileron__right_aileron__failure", "fault_family": "aileron", "fault_label_grouped": "left_aileron__right_aileron__failure", "recovery_variant": "standard"},
        {"fault_label_raw": "left_aileron_failure", "fault_family": "aileron", "fault_label_grouped": "left_aileron_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "no_failure", "fault_family": "normal", "fault_label_grouped": "no_failure", "recovery_variant": "not_applicable"},
        {"fault_label_raw": "no_ground_truth", "fault_family": "unknown", "fault_label_grouped": "no_ground_truth", "recovery_variant": "unknown"},
        {"fault_label_raw": "right_aileron_failure", "fault_family": "aileron", "fault_label_grouped": "right_aileron_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "right_aileron_failure_with_emr_traj", "fault_family": "aileron", "fault_label_grouped": "right_aileron_failure", "recovery_variant": "emr_traj"},
        {"fault_label_raw": "rudder_left_failure", "fault_family": "rudder", "fault_label_grouped": "rudder_left_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "rudder_right_failure", "fault_family": "rudder", "fault_label_grouped": "rudder_right_failure", "recovery_variant": "standard"},
        {"fault_label_raw": "rudder_zero__left_aileron_failure", "fault_family": "multi_surface", "fault_label_grouped": "rudder_zero__left_aileron_failure", "recovery_variant": "standard"}
    ]
)

fault_label_reference.sort_values("fault_label_raw").reset_index(drop = True)

,fault_label_raw,fault_family,fault_label_grouped,recovery_variant
0,both_ailerons_failure,aileron,both_ailerons_failure,standard
1,elevator_failure,elevator,elevator_failure,standard
2,engine_failure,engine,engine_failure,standard
3,engine_failure_with_emr_traj,engine,engine_failure,emr_traj
4,left_aileron__right_aileron__failure,aileron,left_aileron__right_aileron__failure,standard
5,left_aileron_failure,aileron,left_aileron_failure,standard
6,no_failure,normal,no_failure,not_applicable
7,no_ground_truth,unknown,no_ground_truth,unknown
8,right_aileron_failure,aileron,right_aileron_failure,standard
9,right_aileron_failure_with_emr_traj,aileron,right_aileron_failure,emr_traj


In [15]:
inventory_labeled_frame = inventory_frame.merge(
    fault_label_reference,
    on = "fault_label_raw",
    how = "left",
    validate = "many_to_one"
)

inventory_labeled_frame[["flight_name",
                         "fault_label_raw",
                         "fault_family",
                         "fault_label_grouped",
                         "recovery_variant"]].head(20)

,flight_name,fault_label_raw,fault_family,fault_label_grouped,recovery_variant
0,carbonZ_2018-07-18-12-10-11_no_ground_truth,no_ground_truth,unknown,no_ground_truth,unknown
1,carbonZ_2018-07-18-15-53-31_1_engine_failure,engine_failure,engine,engine_failure,standard
2,carbonZ_2018-07-18-15-53-31_2_engine_failure,engine_failure,engine,engine_failure,standard
3,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,engine_failure_with_emr_traj,engine,engine_failure,emr_traj
4,carbonZ_2018-07-18-16-37-39_1_no_failure,no_failure,normal,no_failure,not_applicable
5,carbonZ_2018-07-18-16-37-39_2_engine_failure_w...,engine_failure_with_emr_traj,engine,engine_failure,emr_traj
6,carbonZ_2018-07-30-16-29-45_engine_failure_wit...,engine_failure_with_emr_traj,engine,engine_failure,emr_traj
7,carbonZ_2018-07-30-16-39-00_1_engine_failure,engine_failure,engine,engine_failure,standard
8,carbonZ_2018-07-30-16-39-00_2_engine_failure,engine_failure,engine,engine_failure,standard
9,carbonZ_2018-07-30-16-39-00_3_no_failure,no_failure,normal,no_failure,not_applicable


In [16]:
grouped_fault_distribution = (
    inventory_labeled_frame["fault_label_grouped"]
    .value_counts()
    .sort_index()
    .rename_axis("fault_label_grouped")
    .reset_index(name = "flight_count")
)

grouped_fault_distribution

,fault_label_grouped,flight_count
0,both_ailerons_failure,1
1,elevator_failure,2
2,engine_failure,23
3,left_aileron__right_aileron__failure,1
4,left_aileron_failure,2
5,no_failure,10
6,no_ground_truth,1
7,right_aileron_failure,3
8,rudder_left_failure,1
9,rudder_right_failure,2


In [17]:
failure_status_signal_profile_path = METADATA_ROOT / "processed_failure_status_signal_profile.csv"
failure_status_signal_frame = pd.read_csv(failure_status_signal_profile_path)

first_fault_state_observation_frame = (
    failure_status_signal_frame[
        ["flight_name", "first_non_zero_time"]
    ]
    .dropna(subset = ["first_non_zero_time"])
    .groupby("flight_name", as_index = False)
    .agg(
        first_fault_state_observation_time = ("first_non_zero_time", "min")
    )
)

flight_label_reference_frame = inventory_labeled_frame.copy()

flight_label_reference_frame["supervision_status"] = "usable"
flight_label_reference_frame.loc[
    flight_label_reference_frame["fault_label_raw"] == "no_ground_truth",
    "supervision_status"
] = "exclude_from_supervised_tasks"

flight_label_reference_frame["has_failure_status_file"] = flight_label_reference_frame["flight_name"].isin(
    failure_status_inventory_frame["flight_name"].unique()
)

flight_label_reference_frame["is_fault_flight"] = ~flight_label_reference_frame["fault_label_raw"].isin(
    ["no_failure", "no_ground_truth"]
)

flight_label_reference_frame["fault_state_evidence_source"] = "not_applicable"
flight_label_reference_frame.loc[
    flight_label_reference_frame["is_fault_flight"] &
    flight_label_reference_frame["has_failure_status_file"],
    "fault_state_evidence_source"
] = "failure_status_file"

flight_label_reference_frame.loc[
    flight_label_reference_frame["is_fault_flight"] &
    (~flight_label_reference_frame["has_failure_status_file"]),
    "fault_state_evidence_source"
] = "not_available"

flight_label_reference_frame = flight_label_reference_frame.merge(
    first_fault_state_observation_frame,
    on = "flight_name",
    how = "left",
    validate = "one_to_one"
)

flight_label_reference_frame["fault_onset_timestamp_status"] = "not_applicable"
flight_label_reference_frame.loc[
    flight_label_reference_frame["is_fault_flight"] &
    flight_label_reference_frame["has_failure_status_file"],
    "fault_onset_timestamp_status"
] = "not_confirmed_from_current_artifacts"

flight_label_reference_frame[[
    "flight_name",
    "fault_label_raw",
    "fault_label_grouped",
    "supervision_status",
    "has_failure_status_file",
    "fault_state_evidence_source",
    "first_fault_state_observation_time",
    "fault_onset_timestamp_status",
    "recovery_variant"
]].sort_values(
    ["supervision_status", "fault_label_grouped", "flight_name"]
).reset_index(drop = True)

,flight_name,fault_label_raw,fault_label_grouped,supervision_status,has_failure_status_file,fault_state_evidence_source,first_fault_state_observation_time,fault_onset_timestamp_status,recovery_variant
0,carbonZ_2018-07-18-12-10-11_no_ground_truth,no_ground_truth,no_ground_truth,exclude_from_supervised_tasks,False,not_applicable,NaN,not_applicable,unknown
1,carbonZ_2018-09-11-17-27-13_2_both_ailerons_fa...,both_ailerons_failure,both_ailerons_failure,usable,True,failure_status_file,1.536702e+18,not_confirmed_from_current_artifacts,standard
2,carbonZ_2018-09-11-14-41-51_elevator_failure,elevator_failure,elevator_failure,usable,True,failure_status_file,1.536692e+18,not_confirmed_from_current_artifacts,standard
3,carbonZ_2018-09-11-15-05-11_1_elevator_failure,elevator_failure,elevator_failure,usable,True,failure_status_file,1.536693e+18,not_confirmed_from_current_artifacts,standard
4,carbonZ_2018-07-18-15-53-31_1_engine_failure,engine_failure,engine_failure,usable,True,failure_status_file,1.531944e+18,not_confirmed_from_current_artifacts,standard
5,carbonZ_2018-07-18-15-53-31_2_engine_failure,engine_failure,engine_failure,usable,True,failure_status_file,1.531944e+18,not_confirmed_from_current_artifacts,standard
6,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,engine_failure_with_emr_traj,engine_failure,usable,True,failure_status_file,1.531946e+18,not_confirmed_from_current_artifacts,emr_traj
7,carbonZ_2018-07-18-16-37-39_2_engine_failure_w...,engine_failure_with_emr_traj,engine_failure,usable,True,failure_status_file,1.531947e+18,not_confirmed_from_current_artifacts,emr_traj
8,carbonZ_2018-07-30-16-29-45_engine_failure_wit...,engine_failure_with_emr_traj,engine_failure,usable,True,failure_status_file,1.532133e+18,not_confirmed_from_current_artifacts,emr_traj
9,carbonZ_2018-07-30-16-39-00_1_engine_failure,engine_failure,engine_failure,usable,True,failure_status_file,1.532135e+18,not_confirmed_from_current_artifacts,standard


In [18]:
column_signature_frame = (
    file_profile_frame[["topic_name", "columns"]]
    .drop_duplicates()
    .groupby("topic_name", as_index = False)
    .agg(
        column_signature_count = ("columns", "nunique")
    )
    .sort_values(["column_signature_count", "topic_name"], ascending = [False, True])
    .reset_index(drop = True)
)

column_signature_frame

,topic_name,column_signature_count
0,mavlink-from.csv,8
1,diagnostics.csv,2
2,emergency_responder-traj_file.csv,1
3,failure_status-aileron.csv,1
4,failure_status-elevator.csv,1
5,failure_status-engines.csv,1
6,failure_status-rudder.csv,1
7,mavctrl-path_dev.csv,1
8,mavctrl-rpy.csv,1
9,mavros-battery.csv,1


In [19]:
column_signature_issue_frame = file_profile_frame.loc[
    file_profile_frame["topic_name"].isin(
        column_signature_frame.loc[
            column_signature_frame["column_signature_count"] > 1,
            "topic_name"
        ]
    ),
    ["topic_name", "flight_name", "columns"]
].drop_duplicates().sort_values(
    ["topic_name", "flight_name"]
).reset_index(drop = True)

column_signature_issue_frame

,topic_name,flight_name,columns
0,diagnostics.csv,carbonZ_2018-07-18-12-10-11_no_ground_truth,%time|field.header.seq|field.header.stamp|fiel...
1,diagnostics.csv,carbonZ_2018-07-18-15-53-31_1_engine_failure,%time|field.header.seq|field.header.stamp|fiel...
2,diagnostics.csv,carbonZ_2018-07-18-15-53-31_2_engine_failure,%time|field.header.seq|field.header.stamp|fiel...
3,diagnostics.csv,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,%time|field.header.seq|field.header.stamp|fiel...
4,diagnostics.csv,carbonZ_2018-07-18-16-37-39_1_no_failure,%time|field.header.seq|field.header.stamp|fiel...
...,...,...,...
89,mavlink-from.csv,carbonZ_2018-10-18-11-04-08_1_engine_failure_w...,%time|field.header.seq|field.header.stamp|fiel...
90,mavlink-from.csv,carbonZ_2018-10-18-11-04-08_2_engine_failure_w...,%time|field.header.seq|field.header.stamp|fiel...
91,mavlink-from.csv,carbonZ_2018-10-18-11-04-35_engine_failure_wit...,%time|field.header.seq|field.header.stamp|fiel...
92,mavlink-from.csv,carbonZ_2018-10-18-11-06-06_engine_failure_wit...,%time|field.header.seq|field.header.stamp|fiel...


In [20]:
def infer_time_scale(max_abs_timestamp: float | None) -> str | None:
    if max_abs_timestamp is None:
        return None

    if max_abs_timestamp >= 1e17:
        return "nanoseconds_like"

    if max_abs_timestamp >= 1e14:
        return "microseconds_like"

    if max_abs_timestamp >= 1e11:
        return "milliseconds_like"

    if max_abs_timestamp >= 1e8:
        return "seconds_like"

    return "unknown"


def scale_divisor(time_scale: str | None) -> float | None:
    if time_scale == "nanoseconds_like":
        return 1e9

    if time_scale == "microseconds_like":
        return 1e6

    if time_scale == "milliseconds_like":
        return 1e3

    if time_scale == "seconds_like":
        return 1.0

    return None


def summarize_time_axis(csv_path: Path, timestamp_column: str) -> dict:
    timestamp_frame = pd.read_csv(csv_path, usecols = [timestamp_column])
    timestamp_series = pd.to_numeric(timestamp_frame[timestamp_column], errors = "coerce").dropna()

    if timestamp_series.empty:
        return {
            "timestamp_min_raw": None,
            "timestamp_max_raw": None,
            "duration_seconds": None,
            "median_step_seconds": None,
            "time_scale": None
        }

    timestamp_min_raw = float(timestamp_series.min())
    timestamp_max_raw = float(timestamp_series.max())

    diffs = timestamp_series.diff().dropna()
    positive_diffs = diffs[diffs > 0]

    max_abs_timestamp = float(timestamp_series.abs().max())
    time_scale = infer_time_scale(max_abs_timestamp)
    divisor = scale_divisor(time_scale)

    duration_seconds = None
    median_step_seconds = None

    if divisor is not None:
        duration_seconds = float((timestamp_max_raw - timestamp_min_raw) / divisor)

        if not positive_diffs.empty:
            median_step_seconds = float(positive_diffs.median() / divisor)

    return {
        "timestamp_min_raw": timestamp_min_raw,
        "timestamp_max_raw": timestamp_max_raw,
        "duration_seconds": duration_seconds,
        "median_step_seconds": median_step_seconds,
        "time_scale": time_scale
    }


eligible_topics = structurally_eligible_topics_frame["topic_name"].sort_values().tolist()

time_axis_records = []

eligible_file_frame = file_profile_frame.loc[
    file_profile_frame["topic_name"].isin(eligible_topics),
    ["flight_name", "file_name", "topic_name", "timestamp_column"]
].drop_duplicates()

for row in eligible_file_frame.to_dict(orient = "records"):
    csv_path = PROCESSED_ROOT / row["flight_name"] / row["file_name"]
    time_summary = summarize_time_axis(csv_path, row["timestamp_column"])

    time_axis_records.append(
        {
            "flight_name": row["flight_name"],
            "topic_name": row["topic_name"],
            "timestamp_column": row["timestamp_column"],
            **time_summary
        }
    )

time_axis_frame = pd.DataFrame(time_axis_records)

topic_time_axis_summary = (
    time_axis_frame
    .groupby("topic_name", as_index = False)
    .agg(
        flight_count = ("flight_name", "nunique"),
        time_scale_count = ("time_scale", "nunique"),
        min_duration_seconds = ("duration_seconds", "min"),
        max_duration_seconds = ("duration_seconds", "max"),
        min_median_step_seconds = ("median_step_seconds", "min"),
        max_median_step_seconds = ("median_step_seconds", "max")
    )
    .sort_values(["flight_count", "topic_name"], ascending = [False, True])
    .reset_index(drop = True)
)

topic_time_axis_summary

,topic_name,flight_count,time_scale_count,min_duration_seconds,max_duration_seconds,min_median_step_seconds,max_median_step_seconds
0,mavros-battery.csv,47,1,26.367437,233.351133,0.046219,0.048794
1,mavros-global_position-compass_hdg.csv,47,1,26.249622,233.152643,0.239359,0.247316
2,mavros-global_position-global.csv,47,1,26.249581,233.152524,0.239281,0.247214
3,mavros-global_position-local.csv,47,1,26.249609,233.152554,0.239299,0.247211
4,mavros-global_position-raw-fix.csv,47,1,26.164052,233.226025,0.196831,0.201330
5,mavros-global_position-raw-gps_vel.csv,47,1,26.164103,233.226035,0.196827,0.201145
6,mavros-global_position-rel_alt.csv,47,1,26.249614,233.152605,0.239439,0.247181
7,mavros-imu-atm_pressure.csv,47,1,26.324063,233.278589,0.098724,0.100068
8,mavros-imu-data.csv,47,1,25.838754,233.163181,0.374825,0.412451
9,mavros-imu-data_raw.csv,47,1,26.324887,233.278116,0.098637,0.100531


In [21]:
fault_label_reference_output_path = METADATA_ROOT / "fault_label_reference.csv"
flight_label_reference_output_path = METADATA_ROOT / "flight_label_reference.csv"
flight_label_reference_parquet_output_path = METADATA_ROOT / "flight_label_reference.parquet"

fault_label_reference.sort_values("fault_label_raw").to_csv(
    fault_label_reference_output_path,
    index = False
)

flight_label_reference_frame.sort_values("flight_name").to_csv(
    flight_label_reference_output_path,
    index = False
)

flight_label_reference_frame.sort_values("flight_name").to_parquet(
    flight_label_reference_parquet_output_path,
    index = False
)

print(f"Saved : {fault_label_reference_output_path}")
print(f"Saved : {flight_label_reference_output_path}")
print(f"Saved : {flight_label_reference_parquet_output_path}")

Saved : /Users/bhavikpatwa/Documents/PROJECTS/RoboFleet Fault Detection/data/alfa/metadata/fault_label_reference.csv
Saved : /Users/bhavikpatwa/Documents/PROJECTS/RoboFleet Fault Detection/data/alfa/metadata/flight_label_reference.csv
Saved : /Users/bhavikpatwa/Documents/PROJECTS/RoboFleet Fault Detection/data/alfa/metadata/flight_label_reference.parquet


In [22]:
flight_time_bounds = (
    time_axis_frame
    .groupby("flight_name", as_index = False)
    .agg(
        flight_time_min_raw = ("timestamp_min_raw", "min"),
        flight_time_max_raw = ("timestamp_max_raw", "max")
    )
)

failure_status_timing_frame = (
    failure_status_signal_frame[
        [
            "flight_name",
            "fault_label_raw",
            "topic_name",
            "column_name",
            "non_zero_count",
            "first_non_zero_row_index",
            "first_non_zero_time"
        ]
    ]
    .merge(
        failure_status_inventory_frame[
            ["flight_name", "topic_name", "row_count"]
        ],
        on = ["flight_name", "topic_name"],
        how = "left",
        validate = "many_to_one"
    )
    .merge(
        flight_time_bounds,
        on = "flight_name",
        how = "left",
        validate = "many_to_one"
    )
)

failure_status_timing_frame["all_rows_non_zero"] = (
    failure_status_timing_frame["non_zero_count"] == failure_status_timing_frame["row_count"]
)

failure_status_timing_frame["starts_at_first_failure_status_row"] = (
    failure_status_timing_frame["first_non_zero_row_index"] == 0
)

failure_status_timing_frame["first_non_zero_within_flight_bounds"] = (
    failure_status_timing_frame["first_non_zero_time"].ge(failure_status_timing_frame["flight_time_min_raw"]) &
    failure_status_timing_frame["first_non_zero_time"].le(failure_status_timing_frame["flight_time_max_raw"])
)

failure_status_timing_frame["relative_position_in_flight"] = (
    (failure_status_timing_frame["first_non_zero_time"] - failure_status_timing_frame["flight_time_min_raw"]) /
    (failure_status_timing_frame["flight_time_max_raw"] - failure_status_timing_frame["flight_time_min_raw"])
)

timing_check_summary = pd.DataFrame(
    [
        {
            "check_name": "failure_status_rows_all_non_zero",
            "flight_count": int(failure_status_timing_frame["all_rows_non_zero"].sum())
        },
        {
            "check_name": "failure_status_starts_with_non_zero_row",
            "flight_count": int(failure_status_timing_frame["starts_at_first_failure_status_row"].sum())
        },
        {
            "check_name": "first_non_zero_within_flight_bounds",
            "flight_count": int(failure_status_timing_frame["first_non_zero_within_flight_bounds"].sum())
        }
    ]
)

timing_check_summary

,check_name,flight_count
0,failure_status_rows_all_non_zero,37
1,failure_status_starts_with_non_zero_row,37
2,first_non_zero_within_flight_bounds,37


In [23]:
failure_status_timing_frame[
    [
        "flight_name",
        "fault_label_raw",
        "topic_name",
        "row_count",
        "non_zero_count",
        "all_rows_non_zero",
        "starts_at_first_failure_status_row",
        "first_non_zero_within_flight_bounds",
        "relative_position_in_flight"
    ]
].sort_values(["topic_name", "flight_name"]).reset_index(drop = True)

,flight_name,fault_label_raw,topic_name,row_count,non_zero_count,all_rows_non_zero,starts_at_first_failure_status_row,first_non_zero_within_flight_bounds,relative_position_in_flight
0,carbonZ_2018-09-11-14-52-54_left_aileron__righ...,left_aileron__right_aileron__failure,failure_status-aileron.csv,257,257,True,True,True,0.450977
1,carbonZ_2018-09-11-17-27-13_1_rudder_zero__lef...,rudder_zero__left_aileron_failure,failure_status-aileron.csv,55,55,True,True,True,0.810790
2,carbonZ_2018-09-11-17-27-13_2_both_ailerons_fa...,both_ailerons_failure,failure_status-aileron.csv,72,72,True,True,True,0.646999
3,carbonZ_2018-09-11-17-55-30_1_right_aileron_fa...,right_aileron_failure,failure_status-aileron.csv,43,43,True,True,True,0.840766
4,carbonZ_2018-09-11-17-55-30_2_left_aileron_fai...,left_aileron_failure,failure_status-aileron.csv,63,63,True,True,True,0.613852
5,carbonZ_2018-10-05-14-34-20_2_right_aileron_fa...,right_aileron_failure_with_emr_traj,failure_status-aileron.csv,20,20,True,True,True,0.938620
6,carbonZ_2018-10-05-14-37-22_2_right_aileron_fa...,right_aileron_failure,failure_status-aileron.csv,143,143,True,True,True,0.506331
7,carbonZ_2018-10-05-14-37-22_3_left_aileron_fai...,left_aileron_failure,failure_status-aileron.csv,49,49,True,True,True,0.750894
8,carbonZ_2018-09-11-14-41-51_elevator_failure,elevator_failure,failure_status-elevator.csv,22,22,True,True,True,0.918038
9,carbonZ_2018-09-11-15-05-11_1_elevator_failure,elevator_failure,failure_status-elevator.csv,26,26,True,True,True,0.833230


In [24]:
if time_axis_frame["time_scale"].dropna().nunique() != 1:
    raise RuntimeError("Multiple timestamp scales detected across eligible topics.")

common_time_scale = time_axis_frame["time_scale"].dropna().iloc[0]
common_time_divisor = scale_divisor(common_time_scale)

flight_time_alignment_frame = (
    time_axis_frame
    .groupby("flight_name", as_index = False)
    .agg(
        topic_count = ("topic_name", "nunique"),
        timestamp_min_raw_min = ("timestamp_min_raw", "min"),
        timestamp_min_raw_max = ("timestamp_min_raw", "max"),
        timestamp_max_raw_min = ("timestamp_max_raw", "min"),
        timestamp_max_raw_max = ("timestamp_max_raw", "max")
    )
)

flight_time_alignment_frame["topic_start_spread_seconds"] = (
    (flight_time_alignment_frame["timestamp_min_raw_max"] - flight_time_alignment_frame["timestamp_min_raw_min"]) /
    common_time_divisor
)

flight_time_alignment_frame["topic_end_spread_seconds"] = (
    (flight_time_alignment_frame["timestamp_max_raw_max"] - flight_time_alignment_frame["timestamp_max_raw_min"]) /
    common_time_divisor
)

flight_time_alignment_frame.sort_values(
    ["topic_start_spread_seconds", "topic_end_spread_seconds", "flight_name"],
    ascending = [False, False, True]
).reset_index(drop = True)

,flight_name,topic_count,timestamp_min_raw_min,timestamp_min_raw_max,timestamp_max_raw_min,timestamp_max_raw_max,topic_start_spread_seconds,topic_end_spread_seconds
0,carbonZ_2018-07-18-16-37-39_1_no_failure,28,1.531946e+18,1.531946e+18,1.531946e+18,1.531946e+18,0.931333,0.481656
1,carbonZ_2018-07-30-16-29-45_engine_failure_wit...,28,1.532133e+18,1.532133e+18,1.532133e+18,1.532133e+18,0.926612,0.347408
2,carbonZ_2018-07-18-16-37-39_2_engine_failure_w...,28,1.531947e+18,1.531947e+18,1.531947e+18,1.531947e+18,0.860417,0.551936
3,carbonZ_2018-07-18-15-53-31_2_engine_failure,28,1.531944e+18,1.531944e+18,1.531944e+18,1.531944e+18,0.750117,0.352643
4,carbonZ_2018-07-30-17-10-45_engine_failure_wit...,28,1.532985e+18,1.532985e+18,1.532985e+18,1.532985e+18,0.535930,1.071229
5,carbonZ_2018-07-30-16-39-00_1_engine_failure,28,1.532135e+18,1.532135e+18,1.532135e+18,1.532135e+18,0.527404,1.084434
6,carbonZ_2018-09-11-14-22-07_1_engine_failure,28,1.536690e+18,1.536690e+18,1.536690e+18,1.536690e+18,0.490723,0.270554
7,carbonZ_2018-07-18-12-10-11_no_ground_truth,28,1.531931e+18,1.531931e+18,1.531931e+18,1.531931e+18,0.488371,1.009969
8,carbonZ_2018-10-05-14-37-22_2_right_aileron_fa...,28,1.538768e+18,1.538768e+18,1.538769e+18,1.538769e+18,0.477613,0.891465
9,carbonZ_2018-10-18-11-03-57_engine_failure_wit...,28,1.539875e+18,1.539875e+18,1.539875e+18,1.539875e+18,0.471871,0.513426


In [25]:
eligible_topic_quality_frame = file_profile_frame.loc[
    file_profile_frame["topic_name"].isin(structurally_eligible_topics_frame["topic_name"]),
    [
        "flight_name",
        "topic_name",
        "timestamp_null_fraction",
        "timestamp_is_monotonic",
        "max_null_fraction"
    ]
].copy()

eligible_topic_quality_summary = (
    eligible_topic_quality_frame
    .groupby("topic_name", as_index = False)
    .agg(
        flight_count = ("flight_name", "nunique"),
        non_monotonic_flight_count = ("timestamp_is_monotonic", lambda s: int(s.eq(False).sum())),
        max_timestamp_null_fraction = ("timestamp_null_fraction", "max"),
        max_value_null_fraction = ("max_null_fraction", "max")
    )
    .sort_values(
        ["non_monotonic_flight_count", "max_timestamp_null_fraction", "max_value_null_fraction", "topic_name"],
        ascending = [False, False, False, True]
    )
    .reset_index(drop = True)
)

eligible_topic_quality_summary

,topic_name,flight_count,non_monotonic_flight_count,max_timestamp_null_fraction,max_value_null_fraction
0,mavros-battery.csv,47,0,0.0,1.0
1,mavros-rc-in.csv,47,0,0.0,1.0
2,mavros-rc-out.csv,47,0,0.0,1.0
3,mavros-setpoint_raw-target_global.csv,47,0,0.0,1.0
4,mavros-state.csv,47,0,0.0,1.0
5,mavros-time_reference.csv,47,0,0.0,1.0
6,mavros-vfr_hud.csv,47,0,0.0,1.0
7,mavros-wind_estimation.csv,47,0,0.0,1.0
8,mavros-global_position-compass_hdg.csv,47,0,0.0,0.0
9,mavros-global_position-global.csv,47,0,0.0,0.0


In [26]:
classification_reference_frame = flight_label_reference_frame.loc[
    flight_label_reference_frame["supervision_status"] == "usable"
].copy()

anomaly_detection_support = pd.DataFrame(
    [
        {
            "target_name": "anomaly_detection",
            "label_name": "no_failure",
            "flight_count": int((classification_reference_frame["fault_label_grouped"] == "no_failure").sum())
        },
        {
            "target_name": "anomaly_detection",
            "label_name": "fault",
            "flight_count": int((classification_reference_frame["fault_label_grouped"] != "no_failure").sum())
        }
    ]
)

grouped_fault_support = (
    classification_reference_frame.loc[
        classification_reference_frame["fault_label_grouped"] != "no_failure",
        "fault_label_grouped"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label_name")
    .reset_index(name = "flight_count")
)

grouped_fault_support.insert(0, "target_name", "fault_classification_grouped")

fault_family_support = (
    classification_reference_frame.loc[
        ~classification_reference_frame["fault_family"].isin(["normal", "unknown"]),
        "fault_family"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("label_name")
    .reset_index(name = "flight_count")
)

fault_family_support.insert(0, "target_name", "fault_classification_family")

target_support_summary = pd.concat(
    [
        anomaly_detection_support,
        grouped_fault_support,
        fault_family_support
    ],
    ignore_index = True
)

target_support_summary

,target_name,label_name,flight_count
0,anomaly_detection,no_failure,10
1,anomaly_detection,fault,36
2,fault_classification_grouped,both_ailerons_failure,1
3,fault_classification_grouped,elevator_failure,2
4,fault_classification_grouped,engine_failure,23
5,fault_classification_grouped,left_aileron__right_aileron__failure,1
6,fault_classification_grouped,left_aileron_failure,2
7,fault_classification_grouped,right_aileron_failure,3
8,fault_classification_grouped,rudder_left_failure,1
9,fault_classification_grouped,rudder_right_failure,2


In [27]:
flight_label_reference_frame[[
    "flight_name",
    "fault_label_raw",
    "fault_label_grouped",
    "fault_family",
    "supervision_status",
    "has_failure_status_file",
    "fault_state_evidence_source",
    "first_fault_state_observation_time",
    "fault_onset_timestamp_status",
    "recovery_variant"
]].sort_values(
    ["supervision_status", "fault_label_grouped", "flight_name"]
).reset_index(drop = True)

,flight_name,fault_label_raw,fault_label_grouped,fault_family,supervision_status,has_failure_status_file,fault_state_evidence_source,first_fault_state_observation_time,fault_onset_timestamp_status,recovery_variant
0,carbonZ_2018-07-18-12-10-11_no_ground_truth,no_ground_truth,no_ground_truth,unknown,exclude_from_supervised_tasks,False,not_applicable,NaN,not_applicable,unknown
1,carbonZ_2018-09-11-17-27-13_2_both_ailerons_fa...,both_ailerons_failure,both_ailerons_failure,aileron,usable,True,failure_status_file,1.536702e+18,not_confirmed_from_current_artifacts,standard
2,carbonZ_2018-09-11-14-41-51_elevator_failure,elevator_failure,elevator_failure,elevator,usable,True,failure_status_file,1.536692e+18,not_confirmed_from_current_artifacts,standard
3,carbonZ_2018-09-11-15-05-11_1_elevator_failure,elevator_failure,elevator_failure,elevator,usable,True,failure_status_file,1.536693e+18,not_confirmed_from_current_artifacts,standard
4,carbonZ_2018-07-18-15-53-31_1_engine_failure,engine_failure,engine_failure,engine,usable,True,failure_status_file,1.531944e+18,not_confirmed_from_current_artifacts,standard
5,carbonZ_2018-07-18-15-53-31_2_engine_failure,engine_failure,engine_failure,engine,usable,True,failure_status_file,1.531944e+18,not_confirmed_from_current_artifacts,standard
6,carbonZ_2018-07-18-16-22-01_engine_failure_wit...,engine_failure_with_emr_traj,engine_failure,engine,usable,True,failure_status_file,1.531946e+18,not_confirmed_from_current_artifacts,emr_traj
7,carbonZ_2018-07-18-16-37-39_2_engine_failure_w...,engine_failure_with_emr_traj,engine_failure,engine,usable,True,failure_status_file,1.531947e+18,not_confirmed_from_current_artifacts,emr_traj
8,carbonZ_2018-07-30-16-29-45_engine_failure_wit...,engine_failure_with_emr_traj,engine_failure,engine,usable,True,failure_status_file,1.532133e+18,not_confirmed_from_current_artifacts,emr_traj
9,carbonZ_2018-07-30-16-39-00_1_engine_failure,engine_failure,engine_failure,engine,usable,True,failure_status_file,1.532135e+18,not_confirmed_from_current_artifacts,standard


In [28]:
eligible_topic_file_frame = file_profile_frame.loc[
    file_profile_frame["topic_name"].isin(structurally_eligible_topics_frame["topic_name"]),
    ["flight_name", "file_name", "topic_name", "timestamp_column"]
].drop_duplicates().sort_values(
    ["topic_name", "flight_name"]
).reset_index(drop = True)

column_profile_records = []

for row in eligible_topic_file_frame.to_dict(orient = "records"):
    csv_path = PROCESSED_ROOT / row["flight_name"] / row["file_name"]
    frame = pd.read_csv(csv_path)

    for column_name in frame.columns:
        series = frame[column_name]

        column_profile_records.append(
            {
                "flight_name": row["flight_name"],
                "topic_name": row["topic_name"],
                "column_name": column_name,
                "is_timestamp_column": column_name == row["timestamp_column"],
                "null_fraction": float(series.isna().mean()),
                "non_null_count": int(series.notna().sum()),
                "is_all_null": bool(series.notna().sum() == 0)
            }
        )

column_profile_frame = pd.DataFrame(column_profile_records)

eligible_column_quality_summary = (
    column_profile_frame
    .groupby(["topic_name", "column_name", "is_timestamp_column"], as_index = False)
    .agg(
        flight_count = ("flight_name", "nunique"),
        all_null_flight_count = ("is_all_null", "sum"),
        max_null_fraction = ("null_fraction", "max"),
        mean_null_fraction = ("null_fraction", "mean")
    )
    .sort_values(
        ["all_null_flight_count", "max_null_fraction", "topic_name", "column_name"],
        ascending = [False, False, True, True]
    )
    .reset_index(drop = True)
)

eligible_column_quality_summary.loc[
    (eligible_column_quality_summary["is_timestamp_column"] == False) &
    (
        (eligible_column_quality_summary["all_null_flight_count"] > 0) |
        (eligible_column_quality_summary["max_null_fraction"] >= 0.95)
    )
].reset_index(drop = True)

,topic_name,column_name,is_timestamp_column,flight_count,all_null_flight_count,max_null_fraction,mean_null_fraction
0,mavros-battery.csv,field.capacity,False,47,47,1.0,1.0
1,mavros-battery.csv,field.charge,False,47,47,1.0,1.0
2,mavros-battery.csv,field.design_capacity,False,47,47,1.0,1.0
3,mavros-battery.csv,field.header.frame_id,False,47,47,1.0,1.0
4,mavros-battery.csv,field.location,False,47,47,1.0,1.0
5,mavros-battery.csv,field.serial_number,False,47,47,1.0,1.0
6,mavros-rc-in.csv,field.header.frame_id,False,47,47,1.0,1.0
7,mavros-rc-out.csv,field.header.frame_id,False,47,47,1.0,1.0
8,mavros-setpoint_raw-target_global.csv,field.header.frame_id,False,47,47,1.0,1.0
9,mavros-state.csv,field.header.frame_id,False,47,47,1.0,1.0


In [29]:
usable_column_reference_frame = eligible_column_quality_summary.loc[
    (eligible_column_quality_summary["is_timestamp_column"] == False) &
    (eligible_column_quality_summary["all_null_flight_count"] == 0) &
    (eligible_column_quality_summary["max_null_fraction"] < 0.95)
].copy()

topic_reference_frame = (
    topic_assessment_frame[
        [
            "topic_name",
            "flight_count",
            "min_row_count",
            "max_row_count",
            "min_column_count",
            "max_column_count",
            "malformed_file_count",
            "malformed_row_total",
            "is_present_in_all_flights",
            "has_stable_column_count",
            "is_structurally_clean"
        ]
    ]
    .merge(
        column_signature_frame,
        on = "topic_name",
        how = "left",
        validate = "one_to_one"
    )
    .merge(
        eligible_topic_quality_summary[
            [
                "topic_name",
                "non_monotonic_flight_count",
                "max_timestamp_null_fraction",
                "max_value_null_fraction"
            ]
        ],
        on = "topic_name",
        how = "left"
    )
    .merge(
        topic_time_axis_summary[
            [
                "topic_name",
                "time_scale_count",
                "min_duration_seconds",
                "max_duration_seconds",
                "min_median_step_seconds",
                "max_median_step_seconds"
            ]
        ],
        on = "topic_name",
        how = "left"
    )
    .merge(
        usable_column_reference_frame.groupby("topic_name", as_index = False).agg(
            usable_feature_column_count = ("column_name", "nunique")
        ),
        on = "topic_name",
        how = "left"
    )
)

topic_reference_frame["column_signature_count"] = (
    topic_reference_frame["column_signature_count"].fillna(0).astype(int)
)

topic_reference_frame["usable_feature_column_count"] = (
    topic_reference_frame["usable_feature_column_count"].fillna(0).astype(int)
)

topic_reference_frame["passes_initial_quality_screen"] = (
    (topic_reference_frame["is_present_in_all_flights"] == True) &
    (topic_reference_frame["has_stable_column_count"] == True) &
    (topic_reference_frame["is_structurally_clean"] == True) &
    (topic_reference_frame["column_signature_count"] == 1)
)

topic_reference_frame["is_failure_status_topic"] = (
    topic_reference_frame["topic_name"].str.startswith("failure_status-")
)

topic_reference_frame["is_recovery_artifact_topic"] = (
    topic_reference_frame["topic_name"] == "emergency_responder-traj_file.csv"
)

topic_reference_frame["is_partial_coverage_topic"] = (
    topic_reference_frame["flight_count"] < len(inventory_frame)
)

topic_reference_frame["exclude_from_baseline_features"] = False
topic_reference_frame["baseline_feature_exclusion_reason"] = ""

topic_reference_frame.loc[
    topic_reference_frame["is_failure_status_topic"] == True,
    ["exclude_from_baseline_features", "baseline_feature_exclusion_reason"]
] = [True, "label_source"]

topic_reference_frame.loc[
    topic_reference_frame["is_recovery_artifact_topic"] == True,
    ["exclude_from_baseline_features", "baseline_feature_exclusion_reason"]
] = [True, "recovery_artifact"]

topic_reference_frame.loc[
    topic_reference_frame["passes_initial_quality_screen"] == False,
    ["exclude_from_baseline_features", "baseline_feature_exclusion_reason"]
] = [True, "fails_initial_quality_screen"]

topic_reference_frame.loc[
    (
        topic_reference_frame["exclude_from_baseline_features"] == False
    ) &
    (
        topic_reference_frame["is_partial_coverage_topic"] == True
    ),
    ["exclude_from_baseline_features", "baseline_feature_exclusion_reason"]
] = [True, "partial_flight_coverage"]

topic_reference_output_path = METADATA_ROOT / "topic_reference.csv"

topic_reference_frame.sort_values("topic_name").to_csv(
    topic_reference_output_path,
    index = False
)

topic_reference_frame.sort_values("topic_name").reset_index(drop = True)

,topic_name,flight_count,min_row_count,max_row_count,min_column_count,max_column_count,malformed_file_count,malformed_row_total,is_present_in_all_flights,has_stable_column_count,...,max_duration_seconds,min_median_step_seconds,max_median_step_seconds,usable_feature_column_count,passes_initial_quality_screen,is_failure_status_topic,is_recovery_artifact_topic,is_partial_coverage_topic,exclude_from_baseline_features,baseline_feature_exclusion_reason
0,diagnostics.csv,47,20,182,116,122,2,55,True,False,...,NaN,NaN,NaN,0,False,False,False,False,True,fails_initial_quality_screen
1,emergency_responder-traj_file.csv,17,1,1,2,2,0,0,False,True,...,NaN,NaN,NaN,0,False,False,True,True,True,fails_initial_quality_screen
2,failure_status-aileron.csv,8,20,257,2,2,0,0,False,True,...,NaN,NaN,NaN,0,False,True,False,True,True,fails_initial_quality_screen
3,failure_status-elevator.csv,2,22,26,2,2,0,0,False,True,...,NaN,NaN,NaN,0,False,True,False,True,True,fails_initial_quality_screen
4,failure_status-engines.csv,23,16,118,2,2,0,0,False,True,...,NaN,NaN,NaN,0,False,True,False,True,True,fails_initial_quality_screen
5,failure_status-rudder.csv,4,19,55,2,2,0,0,False,True,...,NaN,NaN,NaN,0,False,True,False,True,True,fails_initial_quality_screen
6,mavctrl-path_dev.csv,41,1322,11669,4,4,0,0,False,True,...,NaN,NaN,NaN,0,False,False,False,True,True,fails_initial_quality_screen
7,mavctrl-rpy.csv,41,1322,11669,4,4,0,0,False,True,...,NaN,NaN,NaN,0,False,False,False,True,True,fails_initial_quality_screen
8,mavlink-from.csv,47,5284,45967,15,22,47,717988,True,False,...,NaN,NaN,NaN,0,False,False,False,False,True,fails_initial_quality_screen
9,mavros-battery.csv,47,543,4721,16,16,0,0,True,True,...,233.351133,0.046219,0.048794,9,True,False,False,False,False,


In [30]:
eligible_topic_file_frame = file_profile_frame.loc[
    file_profile_frame["topic_name"].isin(
        topic_reference_frame.loc[
            topic_reference_frame["exclude_from_baseline_features"] == False,
            "topic_name"
        ]
    ),
    ["flight_name", "file_name", "topic_name", "timestamp_column"]
].drop_duplicates().sort_values(
    ["topic_name", "flight_name"]
).reset_index(drop = True)

column_dtype_records = []

for row in eligible_topic_file_frame.groupby("topic_name", as_index = False).first().to_dict(orient = "records"):
    csv_path = PROCESSED_ROOT / row["flight_name"] / row["file_name"]
    frame = pd.read_csv(csv_path)

    for column_name in frame.columns:
        column_dtype_records.append(
            {
                "topic_name": row["topic_name"],
                "column_name": column_name,
                "is_timestamp_column": column_name == row["timestamp_column"],
                "representative_dtype": str(frame[column_name].dtype),
                "is_numeric_candidate": bool(pd.api.types.is_numeric_dtype(frame[column_name]))
            }
        )

baseline_feature_column_reference_frame = (
    usable_column_reference_frame
    .merge(
        column_dtype_records and pd.DataFrame(column_dtype_records),
        on = ["topic_name", "column_name", "is_timestamp_column"],
        how = "left",
        validate = "one_to_one"
    )
    .sort_values(["topic_name", "column_name"])
    .reset_index(drop = True)
)

baseline_feature_column_reference_output_path = (
    METADATA_ROOT / "baseline_feature_column_reference.csv"
)

baseline_feature_column_reference_frame.to_csv(
    baseline_feature_column_reference_output_path,
    index = False
)

baseline_feature_column_reference_frame

,topic_name,column_name,is_timestamp_column,flight_count,all_null_flight_count,max_null_fraction,mean_null_fraction,representative_dtype,is_numeric_candidate
0,mavros-battery.csv,field.current,False,47,0,0.0,0.0,float64,True
1,mavros-battery.csv,field.header.seq,False,47,0,0.0,0.0,int64,True
2,mavros-battery.csv,field.header.stamp,False,47,0,0.0,0.0,int64,True
3,mavros-battery.csv,field.percentage,False,47,0,0.0,0.0,float64,True
4,mavros-battery.csv,field.power_supply_health,False,47,0,0.0,0.0,int64,True
...,...,...,...,...,...,...,...,...,...
461,mavros-wind_estimation.csv,field.twist.angular.y,False,47,0,0.0,0.0,float64,True
462,mavros-wind_estimation.csv,field.twist.angular.z,False,47,0,0.0,0.0,float64,True
463,mavros-wind_estimation.csv,field.twist.linear.x,False,47,0,0.0,0.0,float64,True
464,mavros-wind_estimation.csv,field.twist.linear.y,False,47,0,0.0,0.0,float64,True
